# TA-005F — Optimización de Umbrales y Evaluación Final

**Modelo ganador:** DenseNet-121 (TA-004F) — AUC test = 0.8844  
**Objetivo:** Encontrar umbrales óptimos por clase sobre el set de validación y evaluar el modelo final sobre el set de test con esos umbrales.

## 1. Imports

In [1]:
import json
import numpy as np
import pandas as pd
import pydicom
import wandb
import torch
import torch.nn as nn
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from sklearn.metrics import (
    roc_auc_score, f1_score, precision_score, recall_score,
    roc_curve, precision_recall_curve, confusion_matrix,
    classification_report, average_precision_score
)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

Device: cuda
GPU: NVIDIA GeForce RTX 4060


## 2. Configuración

In [2]:
BASE       = Path(r'E:\Taller Integrador I\ModelosIA\Radiografias')
MANIFESTS  = BASE / 'dataset_split' / 'manifests'
VAL_DIR    = BASE / 'dataset_split' / 'val'
TEST_DIR   = BASE / 'dataset_split' / 'test'
NB_DIR     = Path(r'E:\Taller Integrador I\ModelosIA\modelos\Notebooks')
CKPT_DIR   = Path(r'E:\Taller Integrador I\ModelosIA\modelos\checkpoints')

CLASSES     = ['alveolar_pattern', 'bronchial_pattern', 'pleural_effusion', 'cardiomegaly', 'no_finding']
NUM_CLASSES = len(CLASSES)
IMG_SIZE    = 224
BATCH_SIZE  = 16
SEED        = 42

with open(NB_DIR / 'densenet_v2_results.json') as f:
    dn_results = json.load(f)

DROPOUT     = dn_results['params']['dropout']
PREV_AUC    = dn_results['test_auc']
PREV_F1     = dn_results['test_f1']
BASELINE    = dn_results['baseline_auc']

torch.manual_seed(SEED)
np.random.seed(SEED)

print(f'Modelo      : DenseNet-121 (best_densenet_v2.pth)')
print(f'Dropout     : {DROPOUT:.4f}')
print(f'AUC previo  : {PREV_AUC} (threshold=0.5, sin optimizar)')
print(f'F1 previo   : {PREV_F1}')
print(f'Baseline    : {BASELINE}')

Modelo      : DenseNet-121 (best_densenet_v2.pth)
Dropout     : 0.2739
AUC previo  : 0.8844 (threshold=0.5, sin optimizar)
F1 previo   : 0.6077
Baseline    : 0.8902


## 3. W&B Init

In [3]:
wandb.init(
    project='vetxray-cnn',
    entity='dbaylont1-antenor-orrego-private-university',
    name='TA-005F-Threshold-Optimization',
    config={
        'task':       'TA-005F',
        'model':      'densenet121',
        'checkpoint': 'best_densenet_v2.pth',
        'method':     'threshold_optimization_per_class',
        'metric':     'F1',
        'baseline_auc': BASELINE
    },
    tags=['threshold', 'evaluation', 'densenet', 'TA-005F', 'VetXRay']
)
print('W&B run:', wandb.run.url)

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\Pcuser\_netrc.
wandb: Currently logged in as: dbaylont1 (dbaylont1-antenor-orrego-private-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


W&B run: https://wandb.ai/dbaylont1-antenor-orrego-private-university/vetxray-cnn/runs/sozcx40j


## 4. Dataset y DataLoaders

In [4]:
TRANSFORM = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

class VetXRayDataset(Dataset):
    def __init__(self, df, transform, base_dir):
        self.df        = df.reset_index(drop=True)
        self.transform = transform
        self.base_dir  = base_dir
        self.labels    = self._build_labels()

    def _build_labels(self):
        labels = np.zeros((len(self.df), NUM_CLASSES), dtype=np.float32)
        for i, row in self.df.iterrows():
            for j, cls in enumerate(CLASSES):
                if isinstance(row['TAG'], str) and cls in row['TAG']:
                    labels[i, j] = 1.0
        return labels

    def _load_image(self, row):
        if row.get('is_synthetic', False) and pd.notna(row.get('synthetic_path')):
            img = Image.open(row['synthetic_path']).convert('L').resize((IMG_SIZE, IMG_SIZE), Image.LANCZOS)
        else:
            dcm = pydicom.dcmread(str(self.base_dir / row['FileName']))
            arr = dcm.pixel_array.astype(np.float32)
            if hasattr(dcm, 'PhotometricInterpretation') and dcm.PhotometricInterpretation == 'MONOCHROME1':
                arr = arr.max() - arr
            p2, p98 = np.percentile(arr, 2), np.percentile(arr, 98)
            arr = np.clip(arr, p2, p98)
            arr = (arr - p2) / (p98 - p2 + 1e-8)
            img = Image.fromarray((arr * 255).astype(np.uint8)).resize((IMG_SIZE, IMG_SIZE), Image.LANCZOS)
        return Image.merge('RGB', [img, img, img])

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        return self.transform(self._load_image(row)), torch.tensor(self.labels[idx], dtype=torch.float32)

df_val  = pd.read_csv(MANIFESTS / 'val.csv')
df_test = pd.read_csv(MANIFESTS / 'test.csv')

val_loader  = DataLoader(VetXRayDataset(df_val,  TRANSFORM, VAL_DIR),  batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
test_loader = DataLoader(VetXRayDataset(df_test, TRANSFORM, TEST_DIR), batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

print(f'Val : {len(df_val):4d} imágenes — {len(val_loader)} batches')
print(f'Test: {len(df_test):4d} imágenes — {len(test_loader)} batches')

Val : 1063 imágenes — 67 batches
Test:  940 imágenes — 59 batches


## 5. Carga del modelo

In [5]:
model = models.densenet121(weights=None, memory_efficient=True)
in_features = model.classifier.in_features
model.classifier = nn.Sequential(nn.Dropout(DROPOUT), nn.Linear(in_features, NUM_CLASSES))
model.load_state_dict(torch.load(CKPT_DIR / 'best_densenet_v2.pth', map_location=DEVICE))
model = model.eval().to(DEVICE)
print(f'DenseNet-121 cargado — {sum(p.numel() for p in model.parameters()):,} params')

DenseNet-121 cargado — 6,958,981 params


## 6. Inferencia sobre Val y Test

In [6]:
def run_inference(loader):
    all_probs, all_labels = [], []
    with torch.no_grad():
        for imgs, labels in loader:
            with torch.amp.autocast('cuda'):
                probs = torch.sigmoid(model(imgs.to(DEVICE)))
            all_probs.append(probs.cpu().numpy())
            all_labels.append(labels.numpy())
    return np.vstack(all_probs), np.vstack(all_labels)

print('Inferencia en Val...', flush=True)
val_probs, val_labels = run_inference(val_loader)
print(f'Val done  — probs: {val_probs.shape} | labels: {val_labels.shape}')

print('Inferencia en Test...', flush=True)
test_probs, test_labels = run_inference(test_loader)
print(f'Test done — probs: {test_probs.shape} | labels: {test_labels.shape}')

Inferencia en Val...
Val done  — probs: (1063, 5) | labels: (1063, 5)
Inferencia en Test...
Test done — probs: (940, 5) | labels: (940, 5)


## 7. Optimización de umbrales (Val set → maximizar F1 por clase)

In [7]:
thresholds_opt = {}
thresholds_range = np.arange(0.05, 0.95, 0.01)

print(f'{"Clase":<22} {"Thr*":>6} {"F1_val*":>8} {"F1_val(0.5)":>12}')
print('-' * 52)

for i, cls in enumerate(CLASSES):
    best_f1  = -1
    best_thr = 0.5
    for thr in thresholds_range:
        preds = (val_probs[:, i] >= thr).astype(int)
        f1    = f1_score(val_labels[:, i], preds, zero_division=0)
        if f1 > best_f1:
            best_f1  = f1
            best_thr = thr

    thresholds_opt[cls] = round(float(best_thr), 4)
    f1_default = f1_score(val_labels[:, i], (val_probs[:, i] >= 0.5).astype(int), zero_division=0)
    print(f'{cls:<22} {best_thr:>6.2f} {best_f1:>8.4f} {f1_default:>12.4f}')

print(f'\nUmbrales optimizados:')
for cls, thr in thresholds_opt.items():
    print(f'  {cls:<22}: {thr}')

Clase                    Thr*  F1_val*  F1_val(0.5)
----------------------------------------------------
alveolar_pattern         0.56   0.6239       0.6047
bronchial_pattern        0.57   0.4274       0.4060
pleural_effusion         0.87   0.7541       0.6533
cardiomegaly             0.77   0.6024       0.5245
no_finding               0.54   0.8072       0.8023

Umbrales optimizados:
  alveolar_pattern      : 0.56
  bronchial_pattern     : 0.57
  pleural_effusion      : 0.87
  cardiomegaly          : 0.77
  no_finding            : 0.54


## 8. Evaluación en Test: threshold=0.5 vs umbrales optimizados

In [8]:
preds_default = (test_probs >= 0.5).astype(int)
preds_opt     = np.zeros_like(test_probs, dtype=int)
for i, cls in enumerate(CLASSES):
    preds_opt[:, i] = (test_probs[:, i] >= thresholds_opt[cls]).astype(int)

auc_macro   = roc_auc_score(test_labels, test_probs, average='macro')
f1_default  = f1_score(test_labels, preds_default, average='macro', zero_division=0)
f1_opt      = f1_score(test_labels, preds_opt,     average='macro', zero_division=0)
prec_opt    = precision_score(test_labels, preds_opt, average='macro', zero_division=0)
rec_opt     = recall_score(test_labels,   preds_opt, average='macro', zero_division=0)

print('=' * 60)
print('EVALUACIÓN FINAL — Test Set')
print('=' * 60)
print(f'AUC macro       : {auc_macro:.4f}   (independiente del threshold)')
print(f'F1  macro (0.5) : {f1_default:.4f}')
print(f'F1  macro (opt) : {f1_opt:.4f}   (+{f1_opt - f1_default:+.4f})')
print(f'Precision (opt) : {prec_opt:.4f}')
print(f'Recall    (opt) : {rec_opt:.4f}')
print(f'Baseline AUC    : {BASELINE}')
print(f'Delta AUC       : {auc_macro - BASELINE:+.4f}')

print(f'\n{"Clase":<22} {"Thr":>5} {"AUC":>7} {"F1(0.5)":>9} {"F1(opt)":>9} {"Prec":>7} {"Rec":>7}')
print('-' * 70)
for i, cls in enumerate(CLASSES):
    try: auc_c = roc_auc_score(test_labels[:, i], test_probs[:, i])
    except: auc_c = 0.0
    f1_d = f1_score(test_labels[:, i], preds_default[:, i], zero_division=0)
    f1_o = f1_score(test_labels[:, i], preds_opt[:, i],     zero_division=0)
    pr_o = precision_score(test_labels[:, i], preds_opt[:, i], zero_division=0)
    rc_o = recall_score(test_labels[:, i],   preds_opt[:, i], zero_division=0)
    thr  = thresholds_opt[cls]
    print(f'{cls:<22} {thr:>5.2f} {auc_c:>7.4f} {f1_d:>9.4f} {f1_o:>9.4f} {pr_o:>7.4f} {rc_o:>7.4f}')

EVALUACIÓN FINAL — Test Set
AUC macro       : 0.8844   (independiente del threshold)
F1  macro (0.5) : 0.6077
F1  macro (opt) : 0.6358   (++0.0280)
Precision (opt) : 0.5842
Recall    (opt) : 0.7068
Baseline AUC    : 0.8902
Delta AUC       : -0.0058

Clase                    Thr     AUC   F1(0.5)   F1(opt)    Prec     Rec
----------------------------------------------------------------------
alveolar_pattern        0.56  0.8947    0.5889    0.5818  0.4729  0.7559
bronchial_pattern       0.57  0.7803    0.3298    0.3436  0.3077  0.3889
pleural_effusion        0.87  0.9812    0.6914    0.7717  0.7424  0.8033
cardiomegaly            0.77  0.8812    0.6262    0.6778  0.6750  0.6807
no_finding              0.54  0.8848    0.8023    0.8039  0.7231  0.9051


## 9. Curvas ROC por clase

In [9]:
COLORS = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12', '#9b59b6']

fig, ax = plt.subplots(figsize=(10, 8))
for i, cls in enumerate(CLASSES):
    fpr, tpr, _ = roc_curve(test_labels[:, i], test_probs[:, i])
    auc_c = roc_auc_score(test_labels[:, i], test_probs[:, i])
    label = f'{cls.replace("_", " ").title()} (AUC={auc_c:.4f})'
    ax.plot(fpr, tpr, color=COLORS[i], lw=2, label=label)

ax.plot([0,1], [0,1], 'k--', lw=1, alpha=0.5, label='Aleatorio (AUC=0.5)')
ax.set_xlabel('Tasa de Falsos Positivos', fontsize=12)
ax.set_ylabel('Tasa de Verdaderos Positivos', fontsize=12)
ax.set_title(f'Curvas ROC — DenseNet-121 (TA-005F)\nAUC macro = {auc_macro:.4f}', fontsize=13)
ax.legend(loc='lower right', fontsize=9)
ax.set_xlim([-0.01, 1.01]); ax.set_ylim([-0.01, 1.01])
ax.grid(alpha=0.3)
plt.tight_layout()
path_roc = NB_DIR / 'ta005f_roc_curves.png'
plt.savefig(str(path_roc), dpi=100, bbox_inches='tight')
wandb.log({'ROC_curves': wandb.Image(str(path_roc))})
plt.show()
print(f'Guardado: {path_roc.name}')

Guardado: ta005f_roc_curves.png


C:\Users\Pcuser\AppData\Local\Temp\ipykernel_7580\2804165182.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 10. Curvas Precision-Recall por clase

In [10]:
fig, ax = plt.subplots(figsize=(10, 8))
for i, cls in enumerate(CLASSES):
    prec_c, rec_c, _ = precision_recall_curve(test_labels[:, i], test_probs[:, i])
    ap = average_precision_score(test_labels[:, i], test_probs[:, i])
    thr_opt = thresholds_opt[cls]
    label = f'{cls.replace("_", " ").title()} (AP={ap:.4f}, thr*={thr_opt})'
    ax.plot(rec_c, prec_c, color=COLORS[i], lw=2, label=label)

ax.set_xlabel('Recall', fontsize=12)
ax.set_ylabel('Precision', fontsize=12)
ax.set_title('Curvas Precision-Recall — DenseNet-121 (TA-005F)', fontsize=13)
ax.legend(loc='upper right', fontsize=9)
ax.set_xlim([-0.01, 1.01]); ax.set_ylim([-0.01, 1.01])
ax.grid(alpha=0.3)
plt.tight_layout()
path_pr = NB_DIR / 'ta005f_pr_curves.png'
plt.savefig(str(path_pr), dpi=100, bbox_inches='tight')
wandb.log({'PR_curves': wandb.Image(str(path_pr))})
plt.show()
print(f'Guardado: {path_pr.name}')

Guardado: ta005f_pr_curves.png


C:\Users\Pcuser\AppData\Local\Temp\ipykernel_7580\4057653408.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 11. Matrices de confusión por clase (umbral optimizado)

In [11]:
fig, axes = plt.subplots(1, NUM_CLASSES, figsize=(4*NUM_CLASSES, 4))
fig.suptitle('Matrices de Confusión por Clase — Umbral Optimizado', fontsize=13, fontweight='bold')

for i, (cls, ax) in enumerate(zip(CLASSES, axes)):
    cm = confusion_matrix(test_labels[:, i], preds_opt[:, i])
    thr = thresholds_opt[cls]
    im = ax.imshow(cm, interpolation='nearest', cmap='Blues')
    ax.set_title(f'{cls.replace("_", chr(10)).replace(" ", chr(10))}\nthr={thr}', fontsize=8)
    ax.set_xlabel('Predicho', fontsize=8)
    ax.set_ylabel('Real', fontsize=8)
    ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
    ax.set_xticklabels(['Neg', 'Pos'], fontsize=8)
    ax.set_yticklabels(['Neg', 'Pos'], fontsize=8)
    for r in range(2):
        for c in range(2):
            color = 'white' if cm[r,c] > cm.max()/2 else 'black'
            ax.text(c, r, str(cm[r,c]), ha='center', va='center', fontsize=12, color=color, fontweight='bold')

plt.tight_layout()
path_cm = NB_DIR / 'ta005f_confusion_matrices.png'
plt.savefig(str(path_cm), dpi=100, bbox_inches='tight')
wandb.log({'Confusion_matrices': wandb.Image(str(path_cm))})
plt.show()
print(f'Guardado: {path_cm.name}')

Guardado: ta005f_confusion_matrices.png


C:\Users\Pcuser\AppData\Local\Temp\ipykernel_7580\3223050202.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 12. Reporte final completo

In [12]:
label_names = [cls.replace('_', ' ').title() for cls in CLASSES]
print('=== Classification Report (umbral optimizado) ===')
print(classification_report(
    test_labels, preds_opt,
    target_names=label_names,
    zero_division=0
))

print('=== Comparación de thresholds ===')
print(f'{"":<22} {"Default (0.5)":>14} {"Optimizado":>12}')
print(f'{"F1 macro":<22} {f1_default:>14.4f} {f1_opt:>12.4f}  (+{f1_opt-f1_default:.4f})')
print(f'{"Precision macro":<22} {precision_score(test_labels, preds_default, average="macro", zero_division=0):>14.4f} {prec_opt:>12.4f}')
print(f'{"Recall macro":<22} {recall_score(test_labels, preds_default, average="macro", zero_division=0):>14.4f} {rec_opt:>12.4f}')

=== Classification Report (umbral optimizado) ===
                   precision    recall  f1-score   support

 Alveolar Pattern       0.47      0.76      0.58       127
Bronchial Pattern       0.31      0.39      0.34        72
 Pleural Effusion       0.74      0.80      0.77        61
     Cardiomegaly       0.68      0.68      0.68       238
       No Finding       0.72      0.91      0.80       453

        micro avg       0.64      0.78      0.70       951
        macro avg       0.58      0.71      0.64       951
     weighted avg       0.65      0.78      0.71       951
      samples avg       0.66      0.71      0.67       951

=== Comparación de thresholds ===
                        Default (0.5)   Optimizado
F1 macro                       0.6077       0.6358  (+0.0280)
Precision macro                0.4998       0.5842
Recall macro                   0.7813       0.7068


## 13. Guardado de resultados finales

In [14]:
per_class = {}
for i, cls in enumerate(CLASSES):
    try: auc_c = float(roc_auc_score(test_labels[:, i], test_probs[:, i]))
    except: auc_c = 0.0
    per_class[cls] = {
        'threshold': thresholds_opt[cls],
        'auc':       round(auc_c, 4),
        'f1_default': round(float(f1_score(test_labels[:, i], preds_default[:, i], zero_division=0)), 4),
        'f1_opt':     round(float(f1_score(test_labels[:, i], preds_opt[:, i],     zero_division=0)), 4),
        'precision':  round(float(precision_score(test_labels[:, i], preds_opt[:, i], zero_division=0)), 4),
        'recall':     round(float(recall_score(test_labels[:, i],    preds_opt[:, i], zero_division=0)), 4),
        'ap':         round(float(average_precision_score(test_labels[:, i], test_probs[:, i])), 4)
    }

final_results = {
    'task':       'TA-005F',
    'model':      'densenet121',
    'checkpoint': 'best_densenet_v2.pth',
    'test_auc_macro':      round(float(auc_macro), 4),
    'test_f1_default':     round(float(f1_default), 4),
    'test_f1_optimized':   round(float(f1_opt), 4),
    'test_precision_opt':  round(float(prec_opt), 4),
    'test_recall_opt':     round(float(rec_opt), 4),
    'baseline_auc':        BASELINE,
    'delta_auc_vs_baseline': round(float(auc_macro - BASELINE), 4),
    'f1_improvement':      round(float(f1_opt - f1_default), 4),
    'thresholds':          thresholds_opt,
    'per_class':           per_class
}

with open(NB_DIR / 'ta005f_final_results.json', 'w') as f:
    json.dump(final_results, f, indent=4)

wandb.summary.update({
    'test_auc_macro':     final_results['test_auc_macro'],
    'test_f1_default':    final_results['test_f1_default'],
    'test_f1_optimized':  final_results['test_f1_optimized'],
    'test_precision_opt': final_results['test_precision_opt'],
    'test_recall_opt':    final_results['test_recall_opt'],
    'baseline_auc':       BASELINE,
    'delta_auc':          final_results['delta_auc_vs_baseline'],
    'f1_improvement':     final_results['f1_improvement'],
    'thresholds':         thresholds_opt
})

wandb.log({
    'resultados_finales': wandb.Table(
        columns=['clase', 'threshold', 'AUC', 'F1(0.5)', 'F1(opt)', 'Precision', 'Recall', 'AP'],
        data=[
            [cls,
             per_class[cls]['threshold'],
             per_class[cls]['auc'],
             per_class[cls]['f1_default'],
             per_class[cls]['f1_opt'],
             per_class[cls]['precision'],
             per_class[cls]['recall'],
             per_class[cls]['ap']]
            for cls in CLASSES
        ] + [['MACRO', None, round(float(auc_macro),4), round(float(f1_default),4),
               round(float(f1_opt),4), round(float(prec_opt),4), round(float(rec_opt),4), None]]
    )
})

wandb.finish()

print('=== TA-005F Completado ===')
print(f'AUC macro       : {auc_macro:.4f}')
print(f'F1 default(0.5) : {f1_default:.4f}')
print(f'F1 optimizado   : {f1_opt:.4f}   (+{f1_opt-f1_default:.4f})')
print(f'Baseline AUC    : {BASELINE}   delta: {auc_macro-BASELINE:+.4f}')
print(f'\nResultados: Notebooks/ta005f_final_results.json')

baseline_auc,0.8902
delta_auc,-0.0058
f1_improvement,0.028
test_auc_macro,0.8844
test_f1_default,0.6077
test_f1_optimized,0.6358
test_precision_opt,0.5842
test_recall_opt,0.7068


=== TA-005F Completado ===
AUC macro       : 0.8844
F1 default(0.5) : 0.6077
F1 optimizado   : 0.6358   (+0.0280)
Baseline AUC    : 0.8902   delta: -0.0058

Resultados: Notebooks/ta005f_final_results.json


## Conclusión

**TA-005F completado.** El modelo DenseNet-121 (TA-004F) fue evaluado con umbrales optimizados por clase sobre el set de validación, maximizando F1 por clase. Los umbrales optimizados se aplicaron al set de test para la evaluación final.

**Archivos generados:**
- `ta005f_roc_curves.png` — Curvas ROC por clase
- `ta005f_pr_curves.png` — Curvas Precision-Recall por clase
- `ta005f_confusion_matrices.png` — Matrices de confusión
- `ta005f_final_results.json` — Métricas finales + umbrales óptimos

**Siguiente paso:** EN-005 / TA-008 — Integración FastAPI con endpoint `/predict`